In [6]:
import pandas as pd

holiday_data = pd.read_csv('raw data/holiday_list.csv')
outlet_coordinates = pd.read_csv('raw data/outlet_coordinates.csv')
outlet_master = pd.read_csv('raw data/outlet_master.csv')
transaction_data = pd.read_csv('raw data/transactions_history_final.csv')
distributor = pd.read_csv('raw data/distributor_seasonality_details.csv')


In [7]:
from pathlib import Path
import json
import pandas as pd

# Summaries output folder
outdir = Path('summaries')
outdir.mkdir(parents=True, exist_ok=True)

p = Path('raw data')
csvs = sorted(p.glob('*.csv'))

print('Found CSV files:', [str(x) for x in csvs])

dataset_rows = []
column_rows = []

for f in csvs:
    size_bytes = f.stat().st_size
    size_mb = size_bytes / 1024 / 1024
    fname = str(f)
    try:
        df = pd.read_csv(f, low_memory=False)
    except Exception:
        df = pd.read_csv(f, engine='python', low_memory=False)
    rows, cols = df.shape
    dataset_rows.append({'file': fname, 'rows': int(rows), 'cols': int(cols), 'size_mb': round(size_mb, 2)})
    for col in df.columns:
        s = df[col]
        missing = int(s.isna().sum())
        unique = int(s.nunique(dropna=True))
        unique_ratio = round(unique / rows, 4) if rows > 0 else None
        missing_ratio = round(missing / rows, 4) if rows > 0 else None
        dtype = str(s.dtype)
        try:
            sample_vals = s.dropna().unique()[:5].tolist()
        except Exception:
            sample_vals = []
        column_rows.append({
            'file': fname,
            'column': col,
            'dtype': dtype,
            'unique_count': unique,
            'unique_ratio': unique_ratio,
            'missing_count': missing,
            'missing_ratio': missing_ratio,
            'sample_values': json.dumps(sample_vals)
        })

# Write summaries into summaries/ folder
pd.DataFrame(dataset_rows).to_csv(outdir / 'dataset_summary.csv', index=False)
pd.DataFrame(column_rows).to_csv(outdir / 'column_summary.csv', index=False)

print('Wrote', outdir / 'dataset_summary.csv', 'and', outdir / 'column_summary.csv')


Found CSV files: ['raw data\\distributor_seasonality_details.csv', 'raw data\\holiday_list.csv', 'raw data\\outlet_coordinates.csv', 'raw data\\outlet_master.csv', 'raw data\\transactions_history_final.csv']
Wrote summaries\dataset_summary.csv and summaries\column_summary.csv


In [8]:
from pathlib import Path
from collections import Counter
import json
import shutil

import numpy as np
import pandas as pd


# ============================================================
# FOLDERS
# ============================================================

RAW_DIR = Path("raw data")

BRONZE_DIR = Path("processed/bronze")
SILVER_DIR = Path("processed/silver")
REJECTED_DIR = Path("processed/rejected")
ANOMALY_DIR = Path("processed/anomalies")
SUMMARY_DIR = Path("summaries")

for folder in [BRONZE_DIR, SILVER_DIR, REJECTED_DIR, ANOMALY_DIR, SUMMARY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================

NULL_TOKENS = {
    "",
    " ",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "-",
    "--",
    "?",
    "missing"
}

# Sri Lanka rough coordinate boundary
LAT_MIN, LAT_MAX = 5.0, 10.5
LON_MIN, LON_MAX = 79.0, 82.5

EXPECTED_TRANSACTION_YEARS = {2023, 2024, 2025}
EXPECTED_SEASONALITY_YEARS = {2023, 2024, 2025, 2026}

VALID_OUTLET_SIZES = {"Small", "Medium", "Large", "Extra Large"}
VALID_SEASONALITY = {"Favorable", "Moderate", "Un-Favorable"}
VALID_HOLIDAY_TYPES = {"Public", "Bank", "Mercantile", "Poya Day"}

PRICE_MIN_RECORDS_PER_SKU = 20


# ============================================================
# COMMON HELPERS
# ============================================================

def read_csv_safely(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        return pd.read_csv(path, engine="python", low_memory=False)


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df


def normalize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    string_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in string_cols:
        s = df[col].astype("string").str.strip()
        s = s.str.replace(r"\s+", " ", regex=True)

        missing_mask = s.str.lower().isin(NULL_TOKENS)
        df[col] = s.mask(missing_mask, pd.NA)

    return df


def standardize_outlet_size(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    cleaned = cleaned.replace({
        "Extra large": "Extra Large",
        "Nan": pd.NA,
        "None": pd.NA,
        "Null": pd.NA,
        "N/A": pd.NA,
        "Na": pd.NA
    })

    return cleaned


def standardize_outlet_type(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    cleaned = cleaned.replace({
        "Grocry": "Grocery",
        "Grocerry": "Grocery",
        "Grocery ": "Grocery",
        "Bakry": "Bakery",
        "Backery": "Bakery",
        "Pharamacy": "Pharmacy",
        "Pharmcy": "Pharmacy"
    })

    return cleaned


def standardize_seasonality(series: pd.Series) -> pd.Series:
    s = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace("_", "-", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

    return s.replace({
        "favorable": "Favorable",
        "favourable": "Favorable",
        "moderate": "Moderate",
        "unfavorable": "Un-Favorable",
        "un-favorable": "Un-Favorable",
        "un favourable": "Un-Favorable",
        "un favorable": "Un-Favorable"
    })


def standardize_holiday_type(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    return cleaned.replace({
        "Public Holiday": "Public",
        "Bank Holiday": "Bank",
        "Mercantile Holiday": "Mercantile",
        "Poya": "Poya Day",
        "Poya day": "Poya Day"
    })


def add_rejection_reason(df: pd.DataFrame, mask: pd.Series, reason: str) -> None:
    mask = mask.fillna(False)

    if "_dq_rejection_reason" not in df.columns:
        df["_dq_rejection_reason"] = ""

    df.loc[mask, "_dq_rejection_reason"] = df.loc[mask, "_dq_rejection_reason"].apply(
        lambda x: reason if x == "" else f"{x} | {reason}"
    )


def is_valid_sri_lanka_coordinate(lat: pd.Series, lon: pd.Series) -> pd.Series:
    return (
        lat.between(LAT_MIN, LAT_MAX) &
        lon.between(LON_MIN, LON_MAX)
    )


def safe_sample_values(series: pd.Series, n: int = 5) -> str:
    try:
        vals = series.dropna().unique()[:n].tolist()
        return json.dumps(vals, default=str)
    except Exception:
        return "[]"


def create_column_summary(file_label: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = len(df)
    output = []

    for col in df.columns:
        s = df[col]

        output.append({
            "file": file_label,
            "column": col,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_ratio": round(float(s.isna().mean()), 4) if rows else None,
            "unique_count": int(s.nunique(dropna=True)),
            "unique_ratio": round(float(s.nunique(dropna=True) / rows), 4) if rows else None,
            "sample_values": safe_sample_values(s)
        })

    return pd.DataFrame(output)


def create_dataset_summary(file_label: str, df: pd.DataFrame) -> dict:
    return {
        "file": file_label,
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    }


def find_existing_file(possible_names: list[str]) -> Path | None:
    for name in possible_names:
        path = RAW_DIR / name
        if path.exists():
            return path
    return None


def save_rejected(df: pd.DataFrame, file_name: str) -> None:
    out_path = REJECTED_DIR / file_name
    df.to_csv(out_path, index=False)


def save_anomaly(df: pd.DataFrame, file_name: str) -> None:
    out_path = ANOMALY_DIR / file_name
    df.to_csv(out_path, index=False)


# ============================================================
# BRONZE COPY
# ============================================================

def create_bronze_copies() -> None:
    for csv_file in RAW_DIR.glob("*.csv"):
        shutil.copy2(csv_file, BRONZE_DIR / csv_file.name)

    print("Bronze layer created:", BRONZE_DIR)


# ============================================================
# OUTLET MASTER SILVER
# ============================================================

def process_outlet_master() -> tuple[pd.DataFrame, dict]:
    path = find_existing_file(["outlet_master.csv", "outlet_master_final.csv"])

    if path is None:
        raise FileNotFoundError("outlet_master.csv not found.")

    raw = read_csv_safely(path)
    df = clean_column_names(raw)
    df = normalize_string_columns(df)

    df["_dq_rejection_reason"] = ""

    if "Outlet_ID" not in df.columns:
        raise ValueError("Outlet_ID column missing in outlet_master.")

    add_rejection_reason(df, df["Outlet_ID"].isna(), "Missing mandatory field: Outlet_ID")

    if "Outlet_Size" in df.columns:
        df["Outlet_Size"] = standardize_outlet_size(df["Outlet_Size"])

        invalid_size = df["Outlet_Size"].notna() & ~df["Outlet_Size"].isin(VALID_OUTLET_SIZES)

        # Do not reject the outlet only because size is wrong.
        # Set invalid size to missing; size imputation will handle it later.
        df["Outlet_Size_Invalid_Flag"] = invalid_size.astype(int)
        df.loc[invalid_size, "Outlet_Size"] = pd.NA
    else:
        df["Outlet_Size"] = pd.NA
        df["Outlet_Size_Invalid_Flag"] = 0

    df["Outlet_Size_Missing_Flag"] = df["Outlet_Size"].isna().astype(int)

    if "Outlet_Type" in df.columns:
        df["Outlet_Type"] = standardize_outlet_type(df["Outlet_Type"])

    if "Cooler_Count" in df.columns:
        before = df["Cooler_Count"]
        df["Cooler_Count"] = pd.to_numeric(df["Cooler_Count"], errors="coerce")

        invalid_cooler = before.notna() & df["Cooler_Count"].isna()
        negative_cooler = df["Cooler_Count"].notna() & (df["Cooler_Count"] < 0)

        add_rejection_reason(df, invalid_cooler, "Invalid numeric value in Cooler_Count")
        add_rejection_reason(df, negative_cooler, "Negative Cooler_Count")
    else:
        df["Cooler_Count"] = np.nan

    # Duplicate Outlet_ID handling
    duplicate_keep = df.duplicated(subset=["Outlet_ID"], keep="first")
    add_rejection_reason(df, duplicate_keep, "Duplicate business key: Outlet_ID")

    rejected = df[df["_dq_rejection_reason"] != ""].copy()
    silver = df[df["_dq_rejection_reason"] == ""].copy()

    rejected.to_csv(REJECTED_DIR / "outlet_master_rejected.csv", index=False)

    silver = silver.drop(columns=["_dq_rejection_reason"], errors="ignore")
    silver.to_csv(SILVER_DIR / "outlet_master_silver.csv", index=False)

    summary = {
        "dataset": "outlet_master",
        "raw_rows": len(raw),
        "silver_rows": len(silver),
        "rejected_rows": len(rejected),
        "missing_outlet_size_rows_kept_for_imputation": int(silver["Outlet_Size"].isna().sum()),
        "invalid_outlet_size_rows_set_to_missing": int(silver["Outlet_Size_Invalid_Flag"].sum())
    }

    return silver, summary


# ============================================================
# OUTLET COORDINATES SILVER WITH LAT/LON SWAP FIX
# ============================================================

def process_outlet_coordinates() -> tuple[pd.DataFrame, dict]:
    path = find_existing_file(["outlet_coordinates.csv", "outlet_coordinates_final.csv"])

    if path is None:
        raise FileNotFoundError("outlet_coordinates.csv not found.")

    raw = read_csv_safely(path)
    df = clean_column_names(raw)
    df = normalize_string_columns(df)

    df["_dq_rejection_reason"] = ""

    required = ["Outlet_ID", "Latitude", "Longitude"]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"{col} column missing in outlet_coordinates.")

    add_rejection_reason(df, df["Outlet_ID"].isna(), "Missing mandatory field: Outlet_ID")

    df["Latitude_Original"] = df["Latitude"]
    df["Longitude_Original"] = df["Longitude"]

    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

    add_rejection_reason(df, df["Latitude"].isna(), "Missing or invalid Latitude")
    add_rejection_reason(df, df["Longitude"].isna(), "Missing or invalid Longitude")

    original_valid = is_valid_sri_lanka_coordinate(df["Latitude"], df["Longitude"])
    swapped_valid = is_valid_sri_lanka_coordinate(df["Longitude"], df["Latitude"])

    needs_swap = (~original_valid) & swapped_valid

    df["coordinate_corrected_flag"] = needs_swap.astype(int)
    df["coordinate_correction_type"] = np.where(
        needs_swap,
        "lat_lon_swapped",
        "none"
    )

    # Correct swapped coordinates
    lat_copy = df.loc[needs_swap, "Latitude"].copy()
    df.loc[needs_swap, "Latitude"] = df.loc[needs_swap, "Longitude"]
    df.loc[needs_swap, "Longitude"] = lat_copy

    final_valid = is_valid_sri_lanka_coordinate(df["Latitude"], df["Longitude"])

    add_rejection_reason(
        df,
        df["Latitude"].notna() & df["Longitude"].notna() & (~final_valid),
        "Invalid coordinates after swap check"
    )

    duplicate_keep = df.duplicated(subset=["Outlet_ID"], keep="first")
    add_rejection_reason(df, duplicate_keep, "Duplicate business key: Outlet_ID")

    rejected = df[df["_dq_rejection_reason"] != ""].copy()
    silver = df[df["_dq_rejection_reason"] == ""].copy()

    rejected.to_csv(REJECTED_DIR / "outlet_coordinates_rejected.csv", index=False)

    correction_log = silver[silver["coordinate_corrected_flag"] == 1].copy()
    correction_log.to_csv(SUMMARY_DIR / "coordinate_corrections_log.csv", index=False)

    silver = silver.drop(columns=["_dq_rejection_reason"], errors="ignore")
    silver.to_csv(SILVER_DIR / "outlet_coordinates_silver.csv", index=False)

    summary = {
        "dataset": "outlet_coordinates",
        "raw_rows": len(raw),
        "silver_rows": len(silver),
        "rejected_rows": len(rejected),
        "lat_lon_swapped_corrected_rows": int(silver["coordinate_corrected_flag"].sum())
    }

    return silver, summary


# ============================================================
# DISTRIBUTOR SEASONALITY SILVER
# ============================================================

def process_distributor_seasonality() -> tuple[pd.DataFrame, dict]:
    path = find_existing_file([
        "distributor_seasonality.csv",
        "distributor_seasonality_details.csv",
        "distributor_seasonality_final.csv"
    ])

    if path is None:
        raise FileNotFoundError("distributor_seasonality file not found.")

    raw = read_csv_safely(path)
    df = clean_column_names(raw)
    df = normalize_string_columns(df)

    df["_dq_rejection_reason"] = ""

    required = ["Distributor_ID", "Year", "Month", "Seasonality_Index"]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"{col} column missing in distributor seasonality.")

    for col in required:
        add_rejection_reason(df, df[col].isna(), f"Missing mandatory field: {col}")

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["Month"] = pd.to_numeric(df["Month"], errors="coerce")

    add_rejection_reason(df, df["Year"].isna(), "Invalid numeric value in Year")
    add_rejection_reason(df, df["Month"].isna(), "Invalid numeric value in Month")
    add_rejection_reason(df, df["Year"].notna() & ~df["Year"].isin(EXPECTED_SEASONALITY_YEARS), "Year outside expected seasonality range")
    add_rejection_reason(df, df["Month"].notna() & ~df["Month"].between(1, 12), "Month outside range 1-12")

    df["Seasonality_Index"] = standardize_seasonality(df["Seasonality_Index"])

    add_rejection_reason(
        df,
        df["Seasonality_Index"].notna() & ~df["Seasonality_Index"].isin(VALID_SEASONALITY),
        "Invalid Seasonality_Index category"
    )

    duplicate_keep = df.duplicated(subset=["Distributor_ID", "Year", "Month"], keep="first")
    add_rejection_reason(df, duplicate_keep, "Duplicate business key: Distributor_ID-Year-Month")

    df["seasonality_score"] = df["Seasonality_Index"].map({
        "Favorable": 1,
        "Moderate": 0,
        "Un-Favorable": -1
    })

    df["seasonality_multiplier"] = df["Seasonality_Index"].map({
        "Favorable": 1.10,
        "Moderate": 1.00,
        "Un-Favorable": 0.90
    })

    rejected = df[df["_dq_rejection_reason"] != ""].copy()
    silver = df[df["_dq_rejection_reason"] == ""].copy()

    rejected.to_csv(REJECTED_DIR / "distributor_seasonality_rejected.csv", index=False)

    silver = silver.drop(columns=["_dq_rejection_reason"], errors="ignore")
    silver.to_csv(SILVER_DIR / "distributor_seasonality_silver.csv", index=False)

    summary = {
        "dataset": "distributor_seasonality",
        "raw_rows": len(raw),
        "silver_rows": len(silver),
        "rejected_rows": len(rejected)
    }

    return silver, summary


# ============================================================
# HOLIDAY SILVER + MONTHLY HOLIDAY FEATURES
# ============================================================

def process_holidays() -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    path = find_existing_file(["holiday_list.csv", "holiday_list_final.csv"])

    if path is None:
        raise FileNotFoundError("holiday_list.csv not found.")

    raw = read_csv_safely(path)
    df = clean_column_names(raw)
    df = normalize_string_columns(df)

    df["_dq_rejection_reason"] = ""

    required = ["Date", "Holiday_Name", "Holiday_Type"]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"{col} column missing in holiday_list.")

    for col in required:
        add_rejection_reason(df, df[col].isna(), f"Missing mandatory field: {col}")

    df["Date_Parsed"] = pd.to_datetime(df["Date"], errors="coerce")
    add_rejection_reason(df, df["Date_Parsed"].isna(), "Invalid Date value")

    df["Holiday_Name"] = (
        df["Holiday_Name"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    df["Holiday_Type"] = standardize_holiday_type(df["Holiday_Type"])

    add_rejection_reason(
        df,
        df["Holiday_Type"].notna() & ~df["Holiday_Type"].isin(VALID_HOLIDAY_TYPES),
        "Invalid Holiday_Type category"
    )

    # Exact duplicate only: same Date + Holiday_Name + Holiday_Type
    duplicate_keep = df.duplicated(subset=["Date", "Holiday_Name", "Holiday_Type"], keep="first")
    add_rejection_reason(df, duplicate_keep, "Exact duplicate holiday record")

    rejected = df[df["_dq_rejection_reason"] != ""].copy()
    silver = df[df["_dq_rejection_reason"] == ""].copy()

    rejected.to_csv(REJECTED_DIR / "holiday_list_rejected.csv", index=False)

    silver["Date"] = silver["Date_Parsed"].dt.date
    silver["Year"] = silver["Date_Parsed"].dt.year
    silver["Month"] = silver["Date_Parsed"].dt.month
    silver["Year_Month"] = silver["Date_Parsed"].dt.to_period("M").astype(str)
    silver["Day_Of_Week"] = silver["Date_Parsed"].dt.dayofweek

    silver["is_public_holiday"] = (silver["Holiday_Type"] == "Public").astype(int)
    silver["is_bank_holiday"] = (silver["Holiday_Type"] == "Bank").astype(int)
    silver["is_mercantile_holiday"] = (silver["Holiday_Type"] == "Mercantile").astype(int)

    silver["is_poya_day"] = (
        (silver["Holiday_Type"] == "Poya Day") |
        (silver["Holiday_Name"].str.contains("Poya", case=False, na=False))
    ).astype(int)

    silver["is_weekend_holiday"] = silver["Day_Of_Week"].isin([5, 6]).astype(int)
    silver["is_long_weekend_proxy"] = silver["Day_Of_Week"].isin([0, 4]).astype(int)

    festival_keywords = [
        "New Year",
        "Vesak",
        "Christmas",
        "Pongal",
        "Deepavali",
        "Ramazan",
        "Eid",
        "Labour",
        "National Day"
    ]

    pattern = "|".join(festival_keywords)

    silver["is_festive_holiday"] = (
        silver["Holiday_Name"]
        .str.contains(pattern, case=False, na=False)
        .astype(int)
    )

    monthly_features = (
        silver
        .groupby(["Year", "Month", "Year_Month"])
        .agg(
            holiday_date_count=("Date", "nunique"),
            holiday_record_count=("Holiday_Name", "count"),
            public_holiday_count=("is_public_holiday", "sum"),
            bank_holiday_count=("is_bank_holiday", "sum"),
            mercantile_holiday_count=("is_mercantile_holiday", "sum"),
            poya_day_count=("is_poya_day", "sum"),
            weekend_holiday_count=("is_weekend_holiday", "sum"),
            long_weekend_holiday_count=("is_long_weekend_proxy", "sum"),
            festive_holiday_count=("is_festive_holiday", "sum")
        )
        .reset_index()
    )

    monthly_features["holiday_intensity_score"] = (
        monthly_features["public_holiday_count"] * 1.00 +
        monthly_features["mercantile_holiday_count"] * 0.80 +
        monthly_features["bank_holiday_count"] * 0.50 +
        monthly_features["poya_day_count"] * 0.70 +
        monthly_features["long_weekend_holiday_count"] * 0.60 +
        monthly_features["festive_holiday_count"] * 1.20
    )

    silver = silver.drop(columns=["_dq_rejection_reason", "Date_Parsed"], errors="ignore")

    silver.to_csv(SILVER_DIR / "holiday_list_silver.csv", index=False)
    monthly_features.to_csv(SILVER_DIR / "holiday_monthly_features_silver.csv", index=False)

    summary = {
        "dataset": "holiday_list",
        "raw_rows": len(raw),
        "silver_rows": len(silver),
        "rejected_rows": len(rejected),
        "monthly_feature_rows": len(monthly_features)
    }

    return silver, monthly_features, summary


# ============================================================
# TRANSACTIONS SILVER
# ============================================================

def build_sku_price_benchmark(sales_df: pd.DataFrame) -> pd.DataFrame:
    benchmark = (
        sales_df
        .groupby("SKU_ID")["Price_Per_Liter"]
        .agg(
            sku_price_count="count",
            sku_price_median="median",
            sku_price_q1=lambda x: np.percentile(x.dropna(), 25),
            sku_price_q3=lambda x: np.percentile(x.dropna(), 75)
        )
        .reset_index()
    )

    benchmark["sku_price_iqr"] = benchmark["sku_price_q3"] - benchmark["sku_price_q1"]

    # If IQR is almost zero, use median-based tolerance.
    benchmark["sku_price_lower_bound"] = np.where(
        benchmark["sku_price_iqr"] > 0,
        benchmark["sku_price_q1"] - 3 * benchmark["sku_price_iqr"],
        benchmark["sku_price_median"] * 0.75
    )

    benchmark["sku_price_upper_bound"] = np.where(
        benchmark["sku_price_iqr"] > 0,
        benchmark["sku_price_q3"] + 3 * benchmark["sku_price_iqr"],
        benchmark["sku_price_median"] * 1.25
    )

    benchmark["sku_price_lower_bound"] = benchmark["sku_price_lower_bound"].clip(lower=0)

    return benchmark


def process_transactions() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    path = find_existing_file(["transactions_history_final.csv", "transactions_history.csv"])

    if path is None:
        raise FileNotFoundError("transactions_history_final.csv / transactions_history.csv not found.")

    raw = read_csv_safely(path)
    df = clean_column_names(raw)
    df = normalize_string_columns(df)

    df["_dq_rejection_reason"] = ""

    required = [
        "Outlet_ID",
        "Year",
        "Month",
        "Distributor_ID",
        "SKU_ID",
        "Volume_Liters",
        "Total_Bill_Value"
    ]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"{col} column missing in transactions.")

    for col in required:
        add_rejection_reason(df, df[col].isna(), f"Missing mandatory field: {col}")

    # Numeric conversion
    for col in ["Year", "Month", "Volume_Liters", "Total_Bill_Value"]:
        original_not_missing = df[col].notna()
        converted = pd.to_numeric(df[col], errors="coerce")

        add_rejection_reason(
            df,
            original_not_missing & converted.isna(),
            f"Invalid numeric value in {col}"
        )

        df[col] = converted

    add_rejection_reason(df, df["Year"].notna() & ~df["Year"].isin(EXPECTED_TRANSACTION_YEARS), "Year outside expected range 2023-2025")
    add_rejection_reason(df, df["Month"].notna() & ~df["Month"].between(1, 12), "Month outside range 1-12")

    # Exact duplicate rows
    exact_duplicate = df.drop(columns=["_dq_rejection_reason"], errors="ignore").duplicated(keep="first")
    add_rejection_reason(df, exact_duplicate, "Exact duplicate row")

    # First separate truly invalid records
    base_rejected = df[df["_dq_rejection_reason"] != ""].copy()
    valid_base = df[df["_dq_rejection_reason"] == ""].copy()

    # Classify transaction business meaning
    positive_sales_mask = (
        (valid_base["Volume_Liters"] > 0) &
        (valid_base["Total_Bill_Value"] > 0)
    )

    returns_mask = (
        (valid_base["Volume_Liters"] < 0) &
        (valid_base["Total_Bill_Value"] < 0)
    )

    zero_volume_positive_bill_mask = (
        (valid_base["Volume_Liters"] == 0) &
        (valid_base["Total_Bill_Value"] > 0)
    )

    zero_value_positive_volume_mask = (
        (valid_base["Volume_Liters"] > 0) &
        (valid_base["Total_Bill_Value"] == 0)
    )

    zero_zero_mask = (
        (valid_base["Volume_Liters"] == 0) &
        (valid_base["Total_Bill_Value"] == 0)
    )

    sign_mismatch_mask = (
        ((valid_base["Volume_Liters"] < 0) & (valid_base["Total_Bill_Value"] >= 0)) |
        ((valid_base["Volume_Liters"] > 0) & (valid_base["Total_Bill_Value"] < 0))
    )

    sales = valid_base[positive_sales_mask].copy()
    returns = valid_base[returns_mask].copy()

    billing_anomalies = valid_base[
        zero_volume_positive_bill_mask |
        zero_value_positive_volume_mask |
        zero_zero_mask
    ].copy()

    sign_mismatch_rejected = valid_base[sign_mismatch_mask].copy()

    if len(sign_mismatch_rejected) > 0:
        sign_mismatch_rejected["_dq_rejection_reason"] = "Sign mismatch between Volume_Liters and Total_Bill_Value"

    if len(billing_anomalies) > 0:
        billing_anomalies["anomaly_type"] = np.select(
            [
                zero_volume_positive_bill_mask.loc[billing_anomalies.index],
                zero_value_positive_volume_mask.loc[billing_anomalies.index],
                zero_zero_mask.loc[billing_anomalies.index]
            ],
            [
                "zero_volume_positive_bill",
                "positive_volume_zero_bill",
                "zero_volume_zero_bill"
            ],
            default="other_billing_anomaly"
        )

    # Price per liter
    sales["Price_Per_Liter"] = sales["Total_Bill_Value"] / sales["Volume_Liters"]
    returns["Price_Per_Liter"] = returns["Total_Bill_Value"].abs() / returns["Volume_Liters"].abs()

    # SKU-level price benchmark using only positive sales
    price_benchmark = build_sku_price_benchmark(sales)
    price_benchmark.to_csv(SUMMARY_DIR / "sku_price_benchmark.csv", index=False)

    sales = sales.merge(price_benchmark, on="SKU_ID", how="left")

    sales["sku_price_outlier_flag"] = (
        (sales["sku_price_count"] >= PRICE_MIN_RECORDS_PER_SKU) &
        (
            (sales["Price_Per_Liter"] < sales["sku_price_lower_bound"]) |
            (sales["Price_Per_Liter"] > sales["sku_price_upper_bound"])
        )
    ).astype(int)

    price_outliers = sales[sales["sku_price_outlier_flag"] == 1].copy()
    sales_clean = sales[sales["sku_price_outlier_flag"] == 0].copy()

    if len(price_outliers) > 0:
        price_outliers["anomaly_type"] = "sku_level_price_outlier"

    # Possible duplicate transaction key report only, do not remove
    tx_key_cols = ["Outlet_ID", "Year", "Month", "Distributor_ID", "SKU_ID"]

    possible_duplicate_keys = (
        sales_clean
        .groupby(tx_key_cols)
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values("row_count", ascending=False)
    )

    possible_duplicate_keys.to_csv(SUMMARY_DIR / "possible_transaction_duplicate_keys.csv", index=False)

    rejected = pd.concat(
        [base_rejected, sign_mismatch_rejected],
        ignore_index=True
    )

    # Save outputs
    sales_clean.to_csv(SILVER_DIR / "transactions_sales_silver.csv", index=False)
    returns.to_csv(SILVER_DIR / "transactions_returns_silver.csv", index=False)

    billing_anomalies.to_csv(ANOMALY_DIR / "transactions_billing_anomalies.csv", index=False)
    price_outliers.to_csv(ANOMALY_DIR / "transactions_price_outliers.csv", index=False)

    rejected.to_csv(REJECTED_DIR / "transactions_rejected.csv", index=False)

    # Return features for later Gold
    return_features = pd.DataFrame()

    if len(returns) > 0:
        returns["return_liters_abs"] = returns["Volume_Liters"].abs()
        returns["return_value_abs"] = returns["Total_Bill_Value"].abs()

        return_features = (
            returns
            .groupby("Outlet_ID")
            .agg(
                return_record_count=("Outlet_ID", "count"),
                return_liters_abs=("return_liters_abs", "sum"),
                return_value_abs=("return_value_abs", "sum"),
                return_month_count=("Month", "nunique"),
                return_sku_count=("SKU_ID", "nunique")
            )
            .reset_index()
        )

    return_features.to_csv(SILVER_DIR / "transaction_return_features_silver.csv", index=False)

    # Billing anomaly features for later Gold
    billing_anomaly_features = pd.DataFrame()

    if len(billing_anomalies) > 0:
        billing_anomaly_features = (
            billing_anomalies
            .groupby("Outlet_ID")
            .agg(
                billing_anomaly_count=("Outlet_ID", "count"),
                zero_volume_positive_bill_count=("anomaly_type", lambda x: (x == "zero_volume_positive_bill").sum()),
                positive_volume_zero_bill_count=("anomaly_type", lambda x: (x == "positive_volume_zero_bill").sum()),
                zero_volume_zero_bill_count=("anomaly_type", lambda x: (x == "zero_volume_zero_bill").sum())
            )
            .reset_index()
        )

    billing_anomaly_features.to_csv(SILVER_DIR / "transaction_billing_anomaly_features_silver.csv", index=False)

    summary = {
        "dataset": "transactions",
        "raw_rows": len(raw),
        "positive_sales_silver_rows": len(sales_clean),
        "returns_silver_rows": len(returns),
        "billing_anomaly_rows": len(billing_anomalies),
        "price_outlier_rows": len(price_outliers),
        "rejected_rows": len(rejected),
        "negative_return_share_of_raw": round(len(returns) / len(raw), 5) if len(raw) else 0,
        "zero_volume_billing_anomaly_share_of_raw": round(len(billing_anomalies) / len(raw), 5) if len(raw) else 0,
        "sku_price_outlier_share_of_positive_sales": round(len(price_outliers) / max(len(sales), 1), 5)
    }

    return sales_clean, returns, billing_anomalies, rejected, summary


# ============================================================
# SUMMARY WRITER
# ============================================================

def write_summaries(outputs: dict, quality_summaries: list[dict]) -> None:
    dataset_summaries = []
    column_summaries = []

    for label, df in outputs.items():
        dataset_summaries.append(create_dataset_summary(label, df))
        column_summaries.append(create_column_summary(label, df))

    pd.DataFrame(dataset_summaries).to_csv(
        SUMMARY_DIR / "silver_dataset_summary.csv",
        index=False
    )

    if column_summaries:
        pd.concat(column_summaries, ignore_index=True).to_csv(
            SUMMARY_DIR / "silver_column_summary.csv",
            index=False
        )

    pd.DataFrame(quality_summaries).to_csv(
        SUMMARY_DIR / "silver_quality_summary.csv",
        index=False
    )

    # Rejection reason summary
    reason_rows = []

    for rejected_file in REJECTED_DIR.glob("*.csv"):
        r = read_csv_safely(rejected_file)

        if "_dq_rejection_reason" in r.columns and len(r) > 0:
            exploded = (
                r["_dq_rejection_reason"]
                .dropna()
                .astype(str)
                .str.split(" | ", regex=False)
                .explode()
            )

            counts = Counter(exploded)

            for reason, count in counts.items():
                reason_rows.append({
                    "file": rejected_file.name,
                    "reason": reason,
                    "count": count
                })

    pd.DataFrame(reason_rows).to_csv(
        SUMMARY_DIR / "silver_rejection_reason_summary.csv",
        index=False
    )


# ============================================================
# MAIN
# ============================================================

def main():
    print("Starting Silver pipeline...")

    create_bronze_copies()

    quality_summaries = []
    outputs = {}

    outlet_master, summary = process_outlet_master()
    quality_summaries.append(summary)
    outputs["outlet_master_silver.csv"] = outlet_master

    outlet_coordinates, summary = process_outlet_coordinates()
    quality_summaries.append(summary)
    outputs["outlet_coordinates_silver.csv"] = outlet_coordinates

    distributor_seasonality, summary = process_distributor_seasonality()
    quality_summaries.append(summary)
    outputs["distributor_seasonality_silver.csv"] = distributor_seasonality

    holiday_list, holiday_monthly_features, summary = process_holidays()
    quality_summaries.append(summary)
    outputs["holiday_list_silver.csv"] = holiday_list
    outputs["holiday_monthly_features_silver.csv"] = holiday_monthly_features

    sales, returns, billing_anomalies, rejected_tx, summary = process_transactions()
    quality_summaries.append(summary)
    outputs["transactions_sales_silver.csv"] = sales
    outputs["transactions_returns_silver.csv"] = returns
    outputs["transactions_billing_anomalies.csv"] = billing_anomalies
    outputs["transactions_rejected.csv"] = rejected_tx

    write_summaries(outputs, quality_summaries)

    print("\nSilver pipeline completed successfully.")
    print("Bronze files:", BRONZE_DIR)
    print("Silver files:", SILVER_DIR)
    print("Rejected files:", REJECTED_DIR)
    print("Anomaly files:", ANOMALY_DIR)
    print("Summary files:", SUMMARY_DIR)

    print("\nMain Silver outputs:")
    print("-", SILVER_DIR / "outlet_master_silver.csv")
    print("-", SILVER_DIR / "outlet_coordinates_silver.csv")
    print("-", SILVER_DIR / "transactions_sales_silver.csv")
    print("-", SILVER_DIR / "transactions_returns_silver.csv")
    print("-", SILVER_DIR / "holiday_monthly_features_silver.csv")
    print("-", SILVER_DIR / "distributor_seasonality_silver.csv")


if __name__ == "__main__":
    main()

Starting Silver pipeline...
Bronze layer created: processed\bronze

Silver pipeline completed successfully.
Bronze files: processed\bronze
Silver files: processed\silver
Rejected files: processed\rejected
Anomaly files: processed\anomalies
Summary files: summaries

Main Silver outputs:
- processed\silver\outlet_master_silver.csv
- processed\silver\outlet_coordinates_silver.csv
- processed\silver\transactions_sales_silver.csv
- processed\silver\transactions_returns_silver.csv
- processed\silver\holiday_monthly_features_silver.csv
- processed\silver\distributor_seasonality_silver.csv


In [9]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ============================================================
# FOLDERS
# ============================================================

SILVER_DIR = Path("processed/silver")
ANOMALY_DIR = Path("processed/anomalies")
GOLD_DIR = Path("processed/gold")
SUMMARY_DIR = Path("summaries")

GOLD_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
LOW_CONFIDENCE_THRESHOLD = 0.80


# ============================================================
# HELPERS
# ============================================================

def read_csv_safely(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        return pd.read_csv(path, engine="python", low_memory=False)


def safe_mode(x):
    mode_val = x.mode()
    if len(mode_val) > 0:
        return mode_val.iloc[0]
    return np.nan


def find_required_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def fill_missing_numeric(df: pd.DataFrame, cols: list[str], fill_value=0) -> pd.DataFrame:
    df = df.copy()

    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(fill_value)

    return df


def standardize_year_month(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

    if "Month" in df.columns:
        df["Month"] = pd.to_numeric(df["Month"], errors="coerce").astype("Int64")

    return df


# ============================================================
# LOAD SILVER DATA
# ============================================================

outlet_master_path = find_required_file(SILVER_DIR / "outlet_master_silver.csv")
outlet_coordinates_path = find_required_file(SILVER_DIR / "outlet_coordinates_silver.csv")
transactions_sales_path = find_required_file(SILVER_DIR / "transactions_sales_silver.csv")
transactions_returns_path = find_required_file(SILVER_DIR / "transactions_returns_silver.csv")
holiday_monthly_path = find_required_file(SILVER_DIR / "holiday_monthly_features_silver.csv")
seasonality_path = find_required_file(SILVER_DIR / "distributor_seasonality_silver.csv")

billing_anomaly_path = SILVER_DIR / "transaction_billing_anomaly_features_silver.csv"
return_features_path = SILVER_DIR / "transaction_return_features_silver.csv"

om = read_csv_safely(outlet_master_path)
oc = read_csv_safely(outlet_coordinates_path)
sales = read_csv_safely(transactions_sales_path)
returns = read_csv_safely(transactions_returns_path)
holidays = read_csv_safely(holiday_monthly_path)
seasonality = read_csv_safely(seasonality_path)

if billing_anomaly_path.exists():
    billing_features = read_csv_safely(billing_anomaly_path)
else:
    billing_features = pd.DataFrame(columns=["Outlet_ID"])

if return_features_path.exists():
    return_features = read_csv_safely(return_features_path)
else:
    return_features = pd.DataFrame(columns=["Outlet_ID"])

print("Loaded Silver files successfully.")


# ============================================================
# BASIC TYPE CLEANING
# ============================================================

sales = standardize_year_month(sales)
returns = standardize_year_month(returns)
holidays = standardize_year_month(holidays)
seasonality = standardize_year_month(seasonality)

for col in ["Volume_Liters", "Total_Bill_Value", "Price_Per_Liter"]:
    if col in sales.columns:
        sales[col] = pd.to_numeric(sales[col], errors="coerce")

for col in ["Volume_Liters", "Total_Bill_Value", "Price_Per_Liter"]:
    if col in returns.columns:
        returns[col] = pd.to_numeric(returns[col], errors="coerce")

if "Cooler_Count" in om.columns:
    om["Cooler_Count"] = pd.to_numeric(om["Cooler_Count"], errors="coerce")

if "Latitude" in oc.columns:
    oc["Latitude"] = pd.to_numeric(oc["Latitude"], errors="coerce")

if "Longitude" in oc.columns:
    oc["Longitude"] = pd.to_numeric(oc["Longitude"], errors="coerce")


# ============================================================
# CREATE MONTHLY SALES GOLD TABLE
# ============================================================

monthly_sales = (
    sales
    .groupby(["Outlet_ID", "Year", "Month"])
    .agg(
        monthly_liters=("Volume_Liters", "sum"),
        monthly_bill_value=("Total_Bill_Value", "sum"),
        monthly_sku_count=("SKU_ID", "nunique"),
        monthly_avg_price_per_liter=("Price_Per_Liter", "mean"),
        monthly_transaction_count=("Outlet_ID", "count")
    )
    .reset_index()
)

monthly_sales["Month_Index"] = monthly_sales["Year"] * 12 + monthly_sales["Month"]

latest_month_index = monthly_sales["Month_Index"].max()

monthly_sales["is_recent_3_months"] = (
    monthly_sales["Month_Index"] >= latest_month_index - 2
).astype(int)

monthly_sales["is_recent_6_months"] = (
    monthly_sales["Month_Index"] >= latest_month_index - 5
).astype(int)


# ============================================================
# ADD HOLIDAY FEATURES TO MONTHLY SALES
# ============================================================

monthly_sales = monthly_sales.merge(
    holidays,
    on=["Year", "Month"],
    how="left"
)

holiday_cols = [
    "holiday_date_count",
    "holiday_record_count",
    "public_holiday_count",
    "bank_holiday_count",
    "mercantile_holiday_count",
    "poya_day_count",
    "weekend_holiday_count",
    "long_weekend_holiday_count",
    "festive_holiday_count",
    "holiday_intensity_score"
]

monthly_sales = fill_missing_numeric(monthly_sales, holiday_cols, 0)


# ============================================================
# ADD SEASONALITY TO MONTHLY SALES
# ============================================================

# Main distributor per outlet from sales
main_dist = (
    sales
    .groupby("Outlet_ID")
    .agg(
        Distributor_ID=("Distributor_ID", safe_mode),
        distributor_count=("Distributor_ID", "nunique")
    )
    .reset_index()
)

monthly_sales = monthly_sales.merge(main_dist, on="Outlet_ID", how="left")

seasonality_cols = [
    "Distributor_ID",
    "Year",
    "Month",
    "Seasonality_Index",
    "seasonality_score",
    "seasonality_multiplier"
]

available_seasonality_cols = [c for c in seasonality_cols if c in seasonality.columns]

monthly_sales = monthly_sales.merge(
    seasonality[available_seasonality_cols],
    on=["Distributor_ID", "Year", "Month"],
    how="left"
)

monthly_sales["Seasonality_Index"] = monthly_sales["Seasonality_Index"].fillna("Moderate")
monthly_sales["seasonality_score"] = pd.to_numeric(
    monthly_sales["seasonality_score"],
    errors="coerce"
).fillna(0)

monthly_sales["seasonality_multiplier"] = pd.to_numeric(
    monthly_sales["seasonality_multiplier"],
    errors="coerce"
).fillna(1.00)


# ============================================================
# CREATE OUTLET-LEVEL SALES FEATURES
# ============================================================

outlet_sales_features = (
    monthly_sales
    .groupby("Outlet_ID")
    .agg(
        avg_monthly_liters=("monthly_liters", "mean"),
        median_monthly_liters=("monthly_liters", "median"),
        max_monthly_liters=("monthly_liters", "max"),
        p75_monthly_liters=("monthly_liters", lambda x: np.percentile(x, 75)),
        p90_monthly_liters=("monthly_liters", lambda x: np.percentile(x, 90)),
        sales_std=("monthly_liters", "std"),
        total_liters=("monthly_liters", "sum"),

        avg_bill_value=("monthly_bill_value", "mean"),
        max_bill_value=("monthly_bill_value", "max"),
        total_bill_value=("monthly_bill_value", "sum"),

        avg_sku_count=("monthly_sku_count", "mean"),
        max_sku_count=("monthly_sku_count", "max"),
        avg_transaction_count=("monthly_transaction_count", "mean"),
        total_transaction_count=("monthly_transaction_count", "sum"),

        avg_price_per_liter=("monthly_avg_price_per_liter", "mean"),

        active_months=("monthly_liters", "count"),

        avg_holiday_intensity_score=("holiday_intensity_score", "mean"),
        max_holiday_intensity_score=("holiday_intensity_score", "max"),
        avg_public_holiday_count=("public_holiday_count", "mean"),
        avg_poya_day_count=("poya_day_count", "mean"),

        avg_seasonality_score=("seasonality_score", "mean"),
        avg_seasonality_multiplier=("seasonality_multiplier", "mean")
    )
    .reset_index()
)

outlet_sales_features["sales_cv"] = (
    outlet_sales_features["sales_std"] /
    outlet_sales_features["avg_monthly_liters"].replace(0, np.nan)
)

outlet_sales_features["sales_cv"] = outlet_sales_features["sales_cv"].replace([np.inf, -np.inf], np.nan)


# ============================================================
# RECENT SALES FEATURES
# ============================================================

recent_3 = (
    monthly_sales[monthly_sales["is_recent_3_months"] == 1]
    .groupby("Outlet_ID")
    .agg(
        recent_3_month_avg_liters=("monthly_liters", "mean"),
        recent_3_month_total_liters=("monthly_liters", "sum")
    )
    .reset_index()
)

recent_6 = (
    monthly_sales[monthly_sales["is_recent_6_months"] == 1]
    .groupby("Outlet_ID")
    .agg(
        recent_6_month_avg_liters=("monthly_liters", "mean"),
        recent_6_month_total_liters=("monthly_liters", "sum")
    )
    .reset_index()
)

january_features = (
    monthly_sales[monthly_sales["Month"] == 1]
    .groupby("Outlet_ID")
    .agg(
        avg_january_liters=("monthly_liters", "mean"),
        max_january_liters=("monthly_liters", "max"),
        avg_january_bill_value=("monthly_bill_value", "mean"),
        avg_january_sku_count=("monthly_sku_count", "mean")
    )
    .reset_index()
)

sku_features = (
    sales
    .groupby("Outlet_ID")
    .agg(
        total_unique_skus=("SKU_ID", "nunique")
    )
    .reset_index()
)

outlet_sales_features = outlet_sales_features.merge(recent_3, on="Outlet_ID", how="left")
outlet_sales_features = outlet_sales_features.merge(recent_6, on="Outlet_ID", how="left")
outlet_sales_features = outlet_sales_features.merge(january_features, on="Outlet_ID", how="left")
outlet_sales_features = outlet_sales_features.merge(sku_features, on="Outlet_ID", how="left")
outlet_sales_features = outlet_sales_features.merge(main_dist, on="Outlet_ID", how="left")


# ============================================================
# RETURN FEATURES
# ============================================================

if len(returns) > 0:
    returns["return_liters_abs"] = returns["Volume_Liters"].abs()
    returns["return_value_abs"] = returns["Total_Bill_Value"].abs()

    return_features_from_rows = (
        returns
        .groupby("Outlet_ID")
        .agg(
            return_record_count=("Outlet_ID", "count"),
            return_liters_abs=("return_liters_abs", "sum"),
            return_value_abs=("return_value_abs", "sum"),
            return_month_count=("Month", "nunique"),
            return_sku_count=("SKU_ID", "nunique")
        )
        .reset_index()
    )
else:
    return_features_from_rows = pd.DataFrame(columns=[
        "Outlet_ID",
        "return_record_count",
        "return_liters_abs",
        "return_value_abs",
        "return_month_count",
        "return_sku_count"
    ])

# Prefer freshly generated return features
return_features = return_features_from_rows


# ============================================================
# BILLING ANOMALY FEATURES
# ============================================================

if not billing_features.empty and "Outlet_ID" in billing_features.columns:
    billing_features = billing_features.copy()
else:
    billing_features = pd.DataFrame(columns=[
        "Outlet_ID",
        "billing_anomaly_count",
        "zero_volume_positive_bill_count",
        "positive_volume_zero_bill_count",
        "zero_volume_zero_bill_count"
    ])


# ============================================================
# BUILD OUTLET GOLD BASE
# ============================================================

outlet_gold = om.merge(
    oc,
    on="Outlet_ID",
    how="left",
    suffixes=("", "_coord")
)

outlet_gold = outlet_gold.merge(
    outlet_sales_features,
    on="Outlet_ID",
    how="left"
)

outlet_gold = outlet_gold.merge(
    return_features,
    on="Outlet_ID",
    how="left"
)

outlet_gold = outlet_gold.merge(
    billing_features,
    on="Outlet_ID",
    how="left"
)


# ============================================================
# MISSING VALUE HANDLING BEFORE OUTLET SIZE IMPUTATION
# ============================================================

# Transaction feature missing means no clean positive sales observed.
sales_feature_cols = [
    "avg_monthly_liters",
    "median_monthly_liters",
    "max_monthly_liters",
    "p75_monthly_liters",
    "p90_monthly_liters",
    "sales_std",
    "total_liters",
    "avg_bill_value",
    "max_bill_value",
    "total_bill_value",
    "avg_sku_count",
    "max_sku_count",
    "avg_transaction_count",
    "total_transaction_count",
    "avg_price_per_liter",
    "active_months",
    "sales_cv",
    "recent_3_month_avg_liters",
    "recent_3_month_total_liters",
    "recent_6_month_avg_liters",
    "recent_6_month_total_liters",
    "avg_january_liters",
    "max_january_liters",
    "avg_january_bill_value",
    "avg_january_sku_count",
    "total_unique_skus",
    "distributor_count",
    "avg_holiday_intensity_score",
    "max_holiday_intensity_score",
    "avg_public_holiday_count",
    "avg_poya_day_count",
    "avg_seasonality_score",
    "avg_seasonality_multiplier"
]

return_feature_cols = [
    "return_record_count",
    "return_liters_abs",
    "return_value_abs",
    "return_month_count",
    "return_sku_count"
]

billing_feature_cols = [
    "billing_anomaly_count",
    "zero_volume_positive_bill_count",
    "positive_volume_zero_bill_count",
    "zero_volume_zero_bill_count"
]

outlet_gold = fill_missing_numeric(outlet_gold, sales_feature_cols, 0)
outlet_gold = fill_missing_numeric(outlet_gold, return_feature_cols, 0)
outlet_gold = fill_missing_numeric(outlet_gold, billing_feature_cols, 0)

outlet_gold["return_ratio_liters"] = (
    outlet_gold["return_liters_abs"] /
    outlet_gold["total_liters"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

outlet_gold["return_ratio_value"] = (
    outlet_gold["return_value_abs"] /
    outlet_gold["total_bill_value"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

outlet_gold["has_return_flag"] = (outlet_gold["return_record_count"] > 0).astype(int)
outlet_gold["has_billing_anomaly_flag"] = (outlet_gold["billing_anomaly_count"] > 0).astype(int)

outlet_gold["Distributor_ID"] = outlet_gold["Distributor_ID"].fillna("Unknown")
outlet_gold["Outlet_Type"] = outlet_gold["Outlet_Type"].fillna("Unknown")


# ============================================================
# OUTLET SIZE IMPUTATION
# ============================================================

target_col = "Outlet_Size"

if target_col not in outlet_gold.columns:
    raise ValueError("Outlet_Size column missing from outlet_gold.")

outlet_gold[target_col] = (
    outlet_gold[target_col]
    .astype("string")
    .str.strip()
    .str.title()
)

outlet_gold[target_col] = outlet_gold[target_col].replace({
    "Nan": pd.NA,
    "None": pd.NA,
    "Null": pd.NA,
    "N/A": pd.NA,
    "Na": pd.NA
})

valid_sizes = {"Small", "Medium", "Large", "Extra Large"}

invalid_size_mask = outlet_gold[target_col].notna() & ~outlet_gold[target_col].isin(valid_sizes)
outlet_gold.loc[invalid_size_mask, target_col] = pd.NA

outlet_gold["Outlet_Size_Imputed"] = 0
outlet_gold["Outlet_Size_Confidence"] = np.nan
outlet_gold["Outlet_Size_Review_Flag"] = "Original"

numeric_features = [
    "Cooler_Count",
    "Latitude",
    "Longitude",

    "avg_monthly_liters",
    "median_monthly_liters",
    "max_monthly_liters",
    "p75_monthly_liters",
    "p90_monthly_liters",
    "sales_std",
    "sales_cv",
    "total_liters",

    "avg_bill_value",
    "max_bill_value",
    "total_bill_value",

    "avg_sku_count",
    "max_sku_count",
    "total_unique_skus",
    "active_months",

    "recent_3_month_avg_liters",
    "recent_6_month_avg_liters",

    "avg_january_liters",
    "max_january_liters",

    "distributor_count",

    "return_record_count",
    "return_liters_abs",
    "return_ratio_liters",

    "billing_anomaly_count",

    "avg_holiday_intensity_score",
    "avg_seasonality_score"
]

categorical_features = [
    "Outlet_Type",
    "Distributor_ID"
]

numeric_features = [c for c in numeric_features if c in outlet_gold.columns]
categorical_features = [c for c in categorical_features if c in outlet_gold.columns]

train = outlet_gold[outlet_gold[target_col].notna()].copy()
missing = outlet_gold[outlet_gold[target_col].isna()].copy()

print("\nOutlet Size Imputation")
print("Training rows:", len(train))
print("Missing rows:", len(missing))
print("\nClass distribution:")
print(train[target_col].value_counts())

if len(missing) > 0 and len(train[target_col].unique()) >= 2:
    X = train[numeric_features + categorical_features]
    y = train[target_col].astype(str)

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    xgb = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=400,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.90,
        colsample_bytree=0.90,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", xgb)
        ]
    )

    min_class_count = train[target_col].value_counts().min()
    n_splits = min(5, min_class_count)

    report_text = "Cross-validation not performed."
    fold_results = []
    oof_pred = np.zeros(len(X), dtype=int)

    if n_splits >= 2:
        cv = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        for fold, (train_idx, val_idx) in enumerate(cv.split(X, y_enc), start=1):
            X_tr = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_tr = y_enc[train_idx]
            y_val = y_enc[val_idx]

            fold_model = clone(model)

            sample_weights = compute_sample_weight(
                class_weight="balanced",
                y=y_tr
            )

            fold_model.fit(
                X_tr,
                y_tr,
                classifier__sample_weight=sample_weights
            )

            pred_val = fold_model.predict(X_val)
            oof_pred[val_idx] = pred_val

            acc = accuracy_score(y_val, pred_val)
            macro_f1 = f1_score(y_val, pred_val, average="macro")

            fold_results.append({
                "fold": fold,
                "accuracy": acc,
                "macro_f1": macro_f1
            })

            print(f"Fold {fold}: Accuracy={acc*100:.2f}% | Macro F1={macro_f1*100:.2f}%")

        fold_results_df = pd.DataFrame(fold_results)
        fold_results_df.to_csv(SUMMARY_DIR / "outlet_size_imputation_cv_results_gold.csv", index=False)

        report_text = classification_report(
            y_enc,
            oof_pred,
            target_names=le.classes_
        )

        cm = confusion_matrix(y_enc, oof_pred)
        cm_df = pd.DataFrame(
            cm,
            index=[f"Actual_{c}" for c in le.classes_],
            columns=[f"Pred_{c}" for c in le.classes_]
        )

        cm_df.to_csv(SUMMARY_DIR / "outlet_size_imputation_confusion_matrix_gold.csv")

        print("\nCross-validation summary:")
        print(f"Accuracy: {fold_results_df['accuracy'].mean()*100:.2f}% ± {fold_results_df['accuracy'].std()*100:.2f}%")
        print(f"Macro F1: {fold_results_df['macro_f1'].mean()*100:.2f}% ± {fold_results_df['macro_f1'].std()*100:.2f}%")

        print("\nClassification report:")
        print(report_text)

    final_sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_enc
    )

    model.fit(
        X,
        y_enc,
        classifier__sample_weight=final_sample_weights
    )

    # Feature importance
    try:
        feature_names = model.named_steps["preprocessor"].get_feature_names_out()
        importances = model.named_steps["classifier"].feature_importances_

        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        }).sort_values("importance", ascending=False)

        importance_df.to_csv(
            SUMMARY_DIR / "outlet_size_imputation_feature_importance_gold.csv",
            index=False
        )

        print("\nTop 20 feature importances:")
        print(importance_df.head(20))

    except Exception as e:
        print("Could not extract feature importance:", e)

    # Predict missing
    X_missing = missing[numeric_features + categorical_features]

    pred_enc = model.predict(X_missing)
    pred_labels = le.inverse_transform(pred_enc)

    pred_proba = model.predict_proba(X_missing)
    confidence = pred_proba.max(axis=1)

    outlet_gold.loc[missing.index, target_col] = pred_labels
    outlet_gold.loc[missing.index, "Outlet_Size_Imputed"] = 1
    outlet_gold.loc[missing.index, "Outlet_Size_Confidence"] = confidence

    outlet_gold.loc[missing.index, "Outlet_Size_Review_Flag"] = np.where(
        confidence < LOW_CONFIDENCE_THRESHOLD,
        "Low Confidence - Review",
        "Imputed OK"
    )

    low_confidence_rows = outlet_gold[
        (outlet_gold["Outlet_Size_Imputed"] == 1) &
        (outlet_gold["Outlet_Size_Confidence"] < LOW_CONFIDENCE_THRESHOLD)
    ].copy()

    low_confidence_rows.to_csv(
        SUMMARY_DIR / "low_confidence_outlet_size_imputations_gold.csv",
        index=False
    )

    with open(SUMMARY_DIR / "outlet_size_imputation_diagnostics_gold.txt", "w", encoding="utf-8") as f:
        f.write("Outlet Size Imputation Diagnostics - Gold Pipeline\n")
        f.write("=================================================\n\n")
        f.write(f"Training rows: {len(train)}\n")
        f.write(f"Missing rows predicted: {len(missing)}\n")
        f.write(f"Low confidence threshold: {LOW_CONFIDENCE_THRESHOLD}\n")
        f.write(f"Low confidence rows: {len(low_confidence_rows)}\n\n")

        f.write("Numeric features used:\n")
        for col in numeric_features:
            f.write(f"- {col}\n")

        f.write("\nCategorical features used:\n")
        for col in categorical_features:
            f.write(f"- {col}\n")

        f.write("\nClass distribution:\n")
        f.write(str(train[target_col].value_counts()))

        f.write("\n\nCross-validation report:\n")
        f.write(report_text)

else:
    print("No missing Outlet_Size values to impute, or not enough target classes.")


# ============================================================
# FINAL GOLD MISSING VALUE HANDLING
# ============================================================

# Important: after imputation, fill remaining non-critical numeric nulls.
final_numeric_cols = outlet_gold.select_dtypes(include=["number"]).columns.tolist()

for col in final_numeric_cols:
    if col in ["Latitude", "Longitude"]:
        # Do not blindly fill coordinates.
        continue

    outlet_gold[col] = outlet_gold[col].replace([np.inf, -np.inf], np.nan)
    outlet_gold[col] = outlet_gold[col].fillna(0)

# Categorical fallback
for col in ["Outlet_Type", "Distributor_ID", "Outlet_Size"]:
    if col in outlet_gold.columns:
        outlet_gold[col] = outlet_gold[col].fillna("Unknown")

# Missing coordinate flags
outlet_gold["missing_coordinate_flag"] = (
    outlet_gold["Latitude"].isna() |
    outlet_gold["Longitude"].isna()
).astype(int)


# ============================================================
# CREATE JANUARY 2026 TARGET CONTEXT FEATURES
# ============================================================

jan_2026_seasonality = seasonality[
    (seasonality["Year"] == 2026) &
    (seasonality["Month"] == 1)
].copy()

jan_2026_seasonality = jan_2026_seasonality[
    ["Distributor_ID", "Seasonality_Index", "seasonality_score", "seasonality_multiplier"]
].rename(columns={
    "Seasonality_Index": "jan_2026_seasonality_index",
    "seasonality_score": "jan_2026_seasonality_score",
    "seasonality_multiplier": "jan_2026_seasonality_multiplier"
})

outlet_gold = outlet_gold.merge(
    jan_2026_seasonality,
    on="Distributor_ID",
    how="left"
)

outlet_gold["jan_2026_seasonality_index"] = outlet_gold["jan_2026_seasonality_index"].fillna("Moderate")
outlet_gold["jan_2026_seasonality_score"] = pd.to_numeric(
    outlet_gold["jan_2026_seasonality_score"],
    errors="coerce"
).fillna(0)

outlet_gold["jan_2026_seasonality_multiplier"] = pd.to_numeric(
    outlet_gold["jan_2026_seasonality_multiplier"],
    errors="coerce"
).fillna(1.00)

# January holiday proxy from historical January data
jan_holiday_history = holidays[holidays["Month"] == 1].copy()

jan_holiday_proxy = (
    jan_holiday_history
    .drop(columns=["Year", "Month", "Year_Month"], errors="ignore")
    .mean(numeric_only=True)
)

for col, value in jan_holiday_proxy.items():
    outlet_gold[f"jan_2026_{col}_proxy"] = value

# Final safety fill
jan_proxy_cols = [c for c in outlet_gold.columns if c.startswith("jan_2026_") and c.endswith("_proxy")]
outlet_gold = fill_missing_numeric(outlet_gold, jan_proxy_cols, 0)


# ============================================================
# SAVE GOLD OUTPUTS
# ============================================================

monthly_sales.to_csv(GOLD_DIR / "monthly_sales_gold.csv", index=False)
outlet_sales_features.to_csv(GOLD_DIR / "outlet_transaction_features_gold.csv", index=False)
outlet_gold.to_csv(GOLD_DIR / "outlet_features_gold.csv", index=False)

# Also save a compact modelling table
modeling_exclude_cols = [
    "Latitude_Original",
    "Longitude_Original"
]

modeling_gold = outlet_gold.drop(columns=modeling_exclude_cols, errors="ignore").copy()

modeling_gold.to_csv(GOLD_DIR / "outlet_modeling_table_gold.csv", index=False)

# Missing value report
missing_report = (
    outlet_gold
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing_report["missing_ratio"] = missing_report["missing_count"] / len(outlet_gold)

missing_report.to_csv(SUMMARY_DIR / "gold_missing_value_report.csv", index=False)

# Gold summary
gold_summary = {
    "outlet_gold_rows": len(outlet_gold),
    "monthly_sales_gold_rows": len(monthly_sales),
    "missing_outlet_size_after_imputation": int(outlet_gold["Outlet_Size"].isna().sum()),
    "outlet_size_imputed_rows": int(outlet_gold["Outlet_Size_Imputed"].sum()),
    "low_confidence_outlet_size_rows": int((outlet_gold["Outlet_Size_Review_Flag"] == "Low Confidence - Review").sum()),
    "missing_coordinate_rows": int(outlet_gold["missing_coordinate_flag"].sum())
}

pd.DataFrame([gold_summary]).to_csv(
    SUMMARY_DIR / "gold_pipeline_summary.csv",
    index=False
)

print("\nGold pipeline completed successfully.")
print("Saved:")
print("-", GOLD_DIR / "monthly_sales_gold.csv")
print("-", GOLD_DIR / "outlet_transaction_features_gold.csv")
print("-", GOLD_DIR / "outlet_features_gold.csv")
print("-", GOLD_DIR / "outlet_modeling_table_gold.csv")
print("-", SUMMARY_DIR / "gold_missing_value_report.csv")
print("-", SUMMARY_DIR / "gold_pipeline_summary.csv")

Loaded Silver files successfully.

Outlet Size Imputation
Training rows: 19804
Missing rows: 196

Class distribution:
Outlet_Size
Small          10272
Medium          5702
Large           2887
Extra Large      943
Name: count, dtype: Int64
Fold 1: Accuracy=99.12% | Macro F1=98.93%
Fold 2: Accuracy=99.27% | Macro F1=98.89%
Fold 3: Accuracy=99.22% | Macro F1=98.73%
Fold 4: Accuracy=99.22% | Macro F1=99.02%
Fold 5: Accuracy=99.09% | Macro F1=98.67%

Cross-validation summary:
Accuracy: 99.18% ± 0.07%
Macro F1: 98.85% ± 0.15%

Classification report:
              precision    recall  f1-score   support

 Extra Large       1.00      0.97      0.98       943
       Large       0.97      1.00      0.98      2887
      Medium       0.99      0.99      0.99      5702
       Small       1.00      1.00      1.00     10272

    accuracy                           0.99     19804
   macro avg       0.99      0.99      0.99     19804
weighted avg       0.99      0.99      0.99     19804


Top 20 featur

In [10]:
from pathlib import Path
import pandas as pd

silver_path = Path('processed/silver/transactions_sales_silver.csv')
rejected_path = Path('processed/rejected/transactions_rejected.csv')

missing = [str(p) for p in (silver_path, rejected_path) if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required file(s): {', '.join(missing)}. "
        "Run the processing cells to generate silver/rejected outputs."
    )

silver = pd.read_csv(silver_path)
rejected = pd.read_csv(rejected_path)

silver_counts = silver['Distributor_ID'].value_counts().rename('silver_count')
rejected_counts = rejected['Distributor_ID'].value_counts().rename('rejected_count')

summary = pd.concat([silver_counts, rejected_counts], axis=1).fillna(0)
summary['percentage'] = (
    (summary['rejected_count'] / summary['silver_count'].replace(0, pd.NA)) * 100
).fillna(0)

summary = (
    summary
    .astype({'silver_count': int, 'rejected_count': int})
    .reset_index()
    .rename(columns={'index': 'Distributor_ID'})
    .sort_values('Distributor_ID')
)

print(summary)


  Distributor_ID  silver_count  rejected_count  percentage
7      DIST_C_01        159064               0         0.0
9      DIST_C_02        147306               0         0.0
8      DIST_C_03        148213               0         0.0
3     DIST_NW_01        231900               0         0.0
4     DIST_NW_02        228374               0         0.0
6      DIST_S_01        159557               0         0.0
5      DIST_S_02        161250               0         0.0
0      DIST_W_01        351011               0         0.0
1      DIST_W_02        350762               0         0.0
2      DIST_W_03        348097               0         0.0
